# **BASELINE USING WEAK SUPERVISION**

In [1]:
# connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **1. Configs**

In [4]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from pathlib import Path

# ============================================
# DEFINE CONFIGS
# ============================================

BASE_PATH = "/content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk"
OUTPUT_DIR = f"{BASE_PATH}/src/training/baseline"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

CONFIG = {
    "model": "Maltehb/danish-bert-botxo",
    "lr": 3e-5,
    "batch_size": 16,
    "weight_decay": 0.01,
    "epochs": 20,
    "warmup_ratio": 0.1,
    "max_length": 512,
    "seed": 62
}

# ============================================
# LOAD DATA
# ============================================

train_df = pd.read_parquet(f"{BASE_PATH}/data/processed/baseline/train_llm_14.9k.parquet")
val_df = pd.read_parquet(f"{BASE_PATH}/data/processed/baseline/val_llm_2.6k.parquet")

print(f"Train: {len(train_df)}, Val: {len(val_df)}")
print(f"Train positives: {train_df['has_claim'].sum()} ({train_df['has_claim'].mean()*100:.1f}%)")

Train: 14850, Val: 2627
Train positives: 14109 (95.0%)


In [7]:
# ============================================
# PREPARE DATASETS
# ============================================

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model'])

def tokenize_function(examples):
    return tokenizer(
        examples['ArticleText'],
        padding='max_length',
        truncation=True,
        max_length=CONFIG['max_length']
    )

# Convert to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df[['ArticleText', 'has_claim']])
val_dataset = Dataset.from_pandas(val_df[['ArticleText', 'has_claim']])

# Tokenize
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Define label format and rename the label column to 'labels' as expected by the Trainer
train_dataset = train_dataset.rename_column("has_claim", "labels")
val_dataset = val_dataset.rename_column("has_claim", "labels")
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

Map:   0%|          | 0/14850 [00:00<?, ? examples/s]

Map:   0%|          | 0/2627 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(CONFIG['model'], num_labels=2)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=CONFIG['epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    learning_rate=CONFIG['lr'],
    weight_decay=CONFIG['weight_decay'],
    warmup_ratio=CONFIG['warmup_ratio'],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    seed=CONFIG['seed'],
    logging_steps=50,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Train
trainer.train()

# ============================================
# SAVE LOGS
# ============================================

logs = trainer.state.log_history
train_logs = [log for log in logs if 'loss' in log and 'eval_loss' not in log]
eval_logs = [log for log in logs if 'eval_loss' in log]

training_logs = []
for t, e in zip(train_logs, eval_logs):
    training_logs.append({
        'epoch': e['epoch'],
        'train_loss': t['loss'],
        'val_loss': e['eval_loss'],
        'val_f1': e['eval_f1'],
        'val_precision': e['eval_precision'],
        'val_recall': e['eval_recall']
    })

logs_df = pd.DataFrame(training_logs)
logs_df.to_csv(f"{OUTPUT_DIR}/training_logs.csv", index=False)

print("\nBest performance:")
print(logs_df.loc[logs_df['val_f1'].idxmax()])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: Maltehb/danish-bert-botxo
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you ex

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.125112,0.135396,0.960030,0.979302,0.963912,0.995192
2,0.153971,0.148850,0.934907,0.965194,0.980968,0.949920
3,0.086001,0.128930,0.964218,0.981297,0.974704,0.987981
4,0.049373,0.186530,0.958888,0.978374,0.977982,0.978766
5,0.066539,0.205887,0.953179,0.975306,0.977465,0.973157
6,0.044811,0.217307,0.961553,0.979972,0.970161,0.989984
7,0.071183,0.180927,0.955843,0.976744,0.977528,0.975962
8,0.020028,0.226823,0.955082,0.976391,0.975220,0.977564
9,0.017281,0.230490,0.962314,0.980384,0.969816,0.991186
10,0.026786,0.254027,0.960792,0.979535,0.971620,0.987580


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Best performance:
epoch            11.000000
train_loss        0.130528
val_loss          0.206283
val_f1            0.981371
val_precision     0.970980
val_recall        0.991987
Name: 10, dtype: float64


## **3. Generate predictions**

In [9]:
# ============================================
# PREDICT
# ============================================

predictions = trainer.predict(val_dataset)
probs = torch.softmax(torch.tensor(predictions.predictions), dim=1).numpy()
pred_labels = np.argmax(predictions.predictions, axis=1)

# Create predictions dataframe
preds_df = val_df[['ArticleKey', 'ArticleText', 'has_claim']].copy()
preds_df['predicted_label'] = pred_labels
preds_df['prob_class_0'] = probs[:, 0]
preds_df['prob_class_1'] = probs[:, 1]
preds_df['VerifiedLabel'] = ''

# Sort by confidence
preds_df = preds_df.sort_values('prob_class_1', ascending=False).reset_index(drop=True)

# Save
preds_df.to_excel(f"{OUTPUT_DIR}/predictions_for_annotation01.xlsx", index=False)

print(f"\Outputs in: {OUTPUT_DIR}")
print(f"Best F1: {logs_df['val_f1'].max():.4f}")

<>:23: SyntaxWarning: invalid escape sequence '\O'
<>:23: SyntaxWarning: invalid escape sequence '\O'
/tmp/ipykernel_896/1518109840.py:23: SyntaxWarning: invalid escape sequence '\O'
  print(f"\Outputs in: {OUTPUT_DIR}")


\Outputs in: /content/drive/MyDrive/Colab Notebooks/misleading-climate-claims-dk/src/training/baseline
Best F1: 0.9814
